In [130]:
import pandas as pd
import os
print(os.getcwd())

# Carga de archivos
productos = pd.read_csv('productos_.csv')
clientes = pd.read_csv('clientes.csv')
ventas = pd.read_csv('ventas.csv')

c:\Users\P3E009-E\Downloads\dataset_examen


PREGUNTA 1 Líneas, tickets y unidades. ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

In [118]:
print("===líneas de venta, tickets distintos y unidades vendidas===")
num_lineas = len(ventas)
num_tickets = ventas['id_ticket'].nunique()
total_unidades = ventas['cantidad'].sum()

print(f"Líneas de venta: {num_lineas}")
print(f"Tickets distintos: {num_tickets}")
print(f"Unidades vendidas: {total_unidades}")

===líneas de venta, tickets distintos y unidades vendidas===
Líneas de venta: 2100
Tickets distintos: 300
Unidades vendidas: 3880


PREGUNTA 2 Tickets por sucursal. ¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?

In [119]:
print("===tickets registrados en cada sucursal===")
tickets_sucursal = ventas.groupby('sucursal')['id_ticket'].nunique().reset_index(name='num_tickets')
print(tickets_sucursal)

sucursal_max = tickets_sucursal.loc[tickets_sucursal['num_tickets'].idxmax()]
print(f"\nLa sucursal con más tickets es: {sucursal_max['sucursal']} con {sucursal_max['num_tickets']} tickets.")

===tickets registrados en cada sucursal===
  sucursal  num_tickets
0  Central          146
1    Norte           83
2      Sur           71

La sucursal con más tickets es: Central con 146 tickets.


PREGUNTA 3 Métodos de pago. ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

In [120]:
print("===tickets pagados con cada método===")
ticket_pago = ventas[['id_ticket', 'metodo_pago']].drop_duplicates()
conteo_pagos = ticket_pago['metodo_pago'].value_counts(dropna=False)
print(conteo_pagos)

===tickets pagados con cada método===
metodo_pago
Efectivo         144
Tarjeta           84
Transferencia     48
NaN               24
Name: count, dtype: int64


PREGUNTA 4 Precios de lista por categoría. ¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.

In [121]:
print("===precio de lista mínimo, máximo y promedio de los productos de cada categoría===")
precios_categoria = productos.groupby('categoria')['precio_lista'].agg(
    minimo='min',
    maximo='max',
    promedio='mean'
).reset_index()

precios_categoria[['minimo', 'maximo', 'promedio']] = precios_categoria[['minimo', 'maximo', 'promedio']].round(2)
print(precios_categoria)

===precio de lista mínimo, máximo y promedio de los productos de cada categoría===
           categoria  minimo  maximo  promedio
0         Accesorios    0.40   22.90      1.62
1            Archivo    0.35    3.95      1.67
2          Escritura    0.40    3.75      1.35
3  Papel y cuadernos    0.28    5.75      1.67


PREGUNTA 5 Unidades por sucursal y categoría. ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de cantidad en ventas.csv.

In [122]:
print("===Unidades vendidas de cada sucursal de cada categoría===")
ventas_prod = ventas.merge(productos, on='id_producto', how='left')

tabla_cruzada = pd.pivot_table(
    ventas_prod,
    values='cantidad',
    index='sucursal',
    columns='categoria',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)
print(tabla_cruzada)
print(f"\nSuma total general de unidades: {ventas_prod['cantidad'].sum()}")

===Unidades vendidas de cada sucursal de cada categoría===
categoria  Accesorios  Archivo  Escritura  Papel y cuadernos  Total
sucursal                                                           
Central           453      481        437                524   1895
Norte             256      264        306                259   1085
Sur               191      223        238                248    900
Total             900      968        981               1031   3880

Suma total general de unidades: 3880


PREGUNTA 6: Importe de ventas por sucursal. ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como cantidad × precio_unitario.

In [123]:
print("===Importe de ventas por sucursal===")
ventas['importe'] = ventas['cantidad'] * ventas['precio_unitario']
importe_sucursal = ventas.groupby('sucursal')['importe'].sum().reset_index()
importe_sucursal['importe'] = importe_sucursal['importe'].round(2)
print(importe_sucursal)

===Importe de ventas por sucursal===
  sucursal  importe
0  Central  3025.04
1    Norte  1635.35
2      Sur  1443.24


PREGUNTA 7: Tamaño de las compras. ¿Cuántos tickets corresponden a compras de hasta 7 unidades, de 8 a 14 unidades y de 15 o más unidades? Sume primero las cantidades de todas las líneas de cada ticket.

In [124]:

ticket_cantidades = ventas.groupby('id_ticket')['cantidad'].sum().reset_index()

def clasificar_tamano(c):
    if c <= 7:
        return 'Hasta 7 unidades'
    elif c <= 14:
        return 'De 8 a 14 unidades'
    else:
        return '15 o más unidades'

ticket_cantidades['tamano'] = ticket_cantidades['cantidad'].apply(clasificar_tamano)
conteo_tamano = ticket_cantidades['tamano'].value_counts()
print(conteo_tamano)

tamano
De 8 a 14 unidades    153
15 o más unidades     108
Hasta 7 unidades       39
Name: count, dtype: int64


PREGUNTA 8: Ventas por mes. ¿Cuántos tickets y qué importe total se registraron en cada mes? ¿En qué mes se alcanzó el mayor importe?

In [125]:
ventas['fecha'] = pd.to_datetime(ventas['fecha'])
ventas['mes'] = ventas['fecha'].dt.to_period('M')

ventas_mes = ventas.groupby('mes').agg(
    tickets=('id_ticket', 'nunique'),
    importe_total=('importe', 'sum')
).reset_index()

ventas_mes['importe_total'] = ventas_mes['importe_total'].round(2)
print(ventas_mes)

mes_mayor = ventas_mes.loc[ventas_mes['importe_total'].idxmax()]
print(f"\nEl mes con mayor importe fue {mes_mayor['mes']} con ${mes_mayor['importe_total']:.2f}.")

        mes  tickets  importe_total
0   2025-11       25         492.51
1   2025-12       25         645.54
2   2026-01       25         447.48
3   2026-02       25         466.36
4   2026-03       25         595.33
5   2026-04       25         505.43
6   2026-05       25         453.94
7   2026-06       25         500.54
8   2026-07       25         499.88
9   2026-08       25         478.79
10  2026-09       25         467.75
11  2026-10       25         550.08

El mes con mayor importe fue 2025-12 con $645.54.


PREGUNTA 9: Producto más vendido por sucursal. ¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

In [126]:
vp_suc = ventas_prod.groupby(['sucursal', 'id_producto', 'producto'])['cantidad'].sum().reset_index()

idx_max = vp_suc.groupby('sucursal')['cantidad'].idxmax()
producto_mas_vendido = vp_suc.loc[idx_max]
print(producto_mas_vendido[['sucursal', 'id_producto', 'producto', 'cantidad']])

    sucursal id_producto                             producto  cantidad
376  Central        P399               Estuche escolar (gris)        17
488    Norte        P141     Marcador de punta fina (naranja)        17
697      Sur        P016  Cuaderno cuadriculado A5 (24 hojas)        11


PREGUNTA 10: Compras por tipo de cliente. Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras? 

In [127]:
ventas_cli = ventas.merge(clientes, left_on='id_cliente', right_on='id', how='left')

compras_tipo_cliente = ventas_cli.groupby('tipo_cliente').agg(
    clientes_distintos=('id_cliente', 'nunique'),
    tickets_generados=('id_ticket', 'nunique'),
    importe_total=('importe', 'sum')
).reset_index()

compras_tipo_cliente['importe_total'] = compras_tipo_cliente['importe_total'].round(2)
print(compras_tipo_cliente)

              tipo_cliente  clientes_distintos  tickets_generados  \
0                  Docente                  16                 46   
1               Estudiante                  72                226   
2  Personal administrativo                  12                 28   

   importe_total  
0         899.74  
1        4688.00  
2         515.89  
